# 🏷️ "THE PRICE IS RIGHT" Capstone Project
## Day 2: Data Pre-processing & LLM-Powered Semantic Rewriting

Welcome to Day 2 of the Week 6 capstone project! Today, we take the raw curated Amazon product data from Day 1 and use an LLM to rewrite noisy, unstructured product descriptions into concise, dense, and structured summaries optimized for machine learning.

### 📅 The 5-Day Progression
1. **DAY 1: Data Curation** — Ingested Amazon reviews, cleaned outliers, balanced price & categories, and pushed raw splits.
2. **DAY 2: Data Pre-processing (Today)** — Rewriting unstructured product descriptions into clean, high-signal summaries using LLMs and high-throughput Batch Mode.
3. **DAY 3: Evaluation, Baselines & Traditional ML** — Establishing benchmark metrics (MAE, RMSE, Median Error) with dummy baselines, linear regression, Random Forests, and XGBoost.
4. **DAY 4: Deep Learning & Frontier LLM Zero-Shot** — Training custom PyTorch neural networks and evaluating zero-shot performance across frontier LLMs (GPT-4o, Claude 3.5 Sonnet, Gemini 1.5/2.0).
5. **DAY 5: Fine-tuning an Open-Source LLM** — Specializing an open-source model (Llama 3 / Mistral) with LoRA / QLoRA to master product valuation.

---

### 💡 The Core Innovation: LLMs as Data Pre-processors
A few years ago, converting chaotic, semi-structured web scrapes into uniform, high-signal product summaries was an unsolved problem. Traditional NLP rules, regex, and keyword extractors were too brittle to handle millions of diverse catalog items. 

Today, modern LLMs perform this semantic restructuring effortlessly. As Ed Donner observed from his experiments on this project, **adding this LLM pre-processing step yielded a dramatic improvement in final model pricing accuracy** compared to training directly on raw scrapes.

> 🔄 **Inference Symmetry**: We apply this exact prompt and formatting logic at inference time when estimating prices for new, unseen products in production.


<table style="margin: 0; text-align: left; border: 1px solid #ddd; border-radius: 8px; padding: 10px;">
    <tr>
        <td style="width: 140px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="130" style="border-radius: 8px; display: block;" />
        </td>
        <td style="padding-left: 15px;">
            <h3 style="color:#1b5e20; margin-top:0;">💡 The Business Value of Data Pre-processing</h3>
            <p style="color:#2e7d32; line-height: 1.5; margin-bottom: 0;">
                Raw web-scraped data is plagued by repetitive seller fluff, SEO keyword stuffing, and irregular formatting. 
                Using an LLM to pre-process descriptions strips away noise, condenses 4,000-character listings into 120-token summaries, and isolates key pricing drivers (brand, category, materials, dimensions).
                <br/><br/>
                Furthermore, by utilizing <strong>asynchronous Batch Mode API inference</strong>, we achieve a <strong>50% cost reduction</strong> across hundreds of thousands of items, making large-scale data transformation commercially viable.
            </p>
        </td>
    </tr>
</table>


### 1. Environment Setup & Core Dependencies

We import our essential libraries:
- **`litellm.completion`**: Unified multi-provider interface that allows switching between Groq, OpenAI, Ollama, Anthropic, etc. without changing calling syntax.
- **`pricer.items.Item`**: Our Pydantic data model representing products, managing training prompts, and handling Hugging Face Hub push/pull.
- **`pricer.batch.Batch`**: Our custom batch orchestrator that automates chunking, uploading, running, and fetching batch jobs.
- **`dotenv`**: Loads API keys (`GROQ_API_KEY`, `HF_TOKEN`) securely from `.env`.


In [6]:
from litellm import completion
from dotenv import load_dotenv
import json
from pricer.batch import Batch
from pricer.items import Item

load_dotenv(override=True)

True

### 2. Selecting Dataset Mode & Cost Breakdown

You can choose between two modes depending on your compute and budget:
- **`LITE_MODE = True` (Recommended for interactive learning)**:
  - Processes **22,000 items** (20,000 train, 1,000 val, 1,000 test).
  - Takes only a few minutes to run.
  - Costs **under $1** on Groq (using Batch Mode).
- **`LITE_MODE = False` (Full scale production)**:
  - Processes all **820,000 items** (800,000 train, 10,000 val, 10,000 test).
  - Takes several hours to process.
  - Costs **under $30** on Groq (50% off manual calls).

> 🆓 **Zero-Cost Option ($0)**: You can follow along by reading the code and output cells without making any API calls. In Day 3, you can load Ed Donner's pre-processed datasets directly from Hugging Face for free!


In [7]:
LITE_MODE = True

### 3. Loading the Curated Dataset from Hugging Face

We load the raw curated dataset produced in Day 1 (`items_raw_lite` or `items_raw_full`) using `Item.from_hub()`.
We combine `train + val + test` into a unified list of 22,000 items to prepare them for batch rewriting:


In [8]:
username = "ed-donner"
dataset = f"{username}/items_raw_lite" if LITE_MODE else f"{username}/items_raw_full"

train, val, test = Item.from_hub(dataset)

items = train + val + test

print(f"Loaded {len(items):,} items")
print(items[0])

Loaded 22,000 items
title='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)' category='Tools_and_Home_Improvement' price=64.3 full='Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)\n[\'From the Manufacturer\', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we\'re the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]\n[\'Interior half only\', \'Requires F58 to complete handle set\', \'Non handed knob style\', \'4" minimum center to center door prep required for this two piece model.\', \'Lifetime Mechanical and Finish Warranty\']\n{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Wei

### 4. Assigning Sequential IDs for Batch Tracking

Let's check the `id` field of an item before assignment (it is initially `None`):


In [11]:
items[2].id

2

#### 🔑 Why do we assign sequential integer IDs?
In Batch Mode, thousands of requests are processed asynchronously across a distributed GPU cluster. Responses return in **random, non-sequential order**.

By setting `item.id = index`, each item receives a unique identifier that is passed as `custom_id` in our batch request. When batch results return hours later, this allows us to map each response back to its exact `Item` object in $O(1)$ constant time!


In [10]:
# Give every item an id

for index, item in enumerate(items):
    item.id = index

---
## 5. System Prompt Engineering: Why Plain Text Over JSON?

Here is our system prompt for rewriting product listings:
```
Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features
```

### ⚖️ The Design Decision: Plain Text vs. JSON / Structured Outputs
Engineers often recommend enforcing JSON or schema-validated **Structured Outputs** (which we dive into in Week 8). Why did we choose a colon-delimited plain text format here instead?
1. **Token Weight & Cost**: JSON formatting carries substantial syntax overhead: curly braces, quotes around keys and values, colons, commas, and escapes. Across **820,000 items**, this formatting boilerplate adds millions of superfluous tokens, increasing API costs and training sequence length.
2. **Speed**: Generating plain text is faster and less prone to JSON formatting edge cases.
3. **Training Signal**: For downstream tabular and NLP regressors, a clean colon-delimited text block provides the exact same high-density semantic signal without the token tax.


In [12]:


SYSTEM_PROMPT = """Create a concise description of a product. Respond only in this format. Do not include part numbers.
Title: Rewritten short precise title
Category: eg Electronics
Brand: Brand name
Description: 1 sentence description
Details: 1 sentence on features"""

### 6. Testing Synchronous Rewriting on a Single Item

Let's inspect the raw text of `items[0]` (the Andover interior knob with deadbolt).
Notice how unstructured it is: 446 tokens containing repetitive warranty disclaimers, disjointed bullet points, and raw Amazon scrape fragments:


In [13]:
print(items[0].full)

Schlage F59 AND 613 Andover Interior Knob with Deadbolt, Oil Rubbed Bronze (Interior Half Only)
['From the Manufacturer', "When you have a Schlage handleset on your front door, you ensure your security as well as your peace of mind. After all, we're the leader in security devices, trusted for over 85 years. All Schlage handlesets are precision engineered, featuring 100% solid"]
['Interior half only', 'Requires F58 to complete handle set', 'Non handed knob style', '4" minimum center to center door prep required for this two piece model.', 'Lifetime Mechanical and Finish Warranty']
{"Material": "Metal", "Brand": "", "Color": "Oil Rubbed Bronze", "Exterior Finish": "Bronze", "Special Feature": "Easy to Install", "Age Range (Description)": "Adult", "Included Components": "Deadbolt, Knob", "Item Weight": "1.5 pounds", "Handle Material": "Bronze", "Package Type": "Standard Packaging", "Unit Count": "1.0 Count", "Number of Items": "1", "Manufacturer": "Schlage", "Product Dimensions": "8.1 x 4

Now let's send this item through `groq/openai/gpt-oss-20b` via LiteLLM using Groq's ultra-fast LPU inference engine:
- Notice how rapidly the response returns!
- Input: **446 tokens** -> Output: **~125 tokens** (including reasoning tokens).
- Cost: **0.011 cents** ($0.00011, or 1/10,000th of a dollar).
- Look at the resulting summary: a crisp title, clean brand, and punchy 1-sentence description and details. The signal-to-noise ratio is vastly superior!


In [14]:
messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="groq/openai/gpt-oss-20b", reasoning_effort="low")

print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


Title: Schlage F59 & 613 Interior Knob with Deadbolt – Oil Rubbed Bronze  
Category: Hardware & Locks  
Brand: Schlage  
Description: A complete interior half‑door set featuring an oil‑rubbed bronze knob with integrated deadbolt for secure and stylish entry.  
Details: Designed for 4" center‑to‑center door prep, it offers easy installation, a lifetime mechanical and finish warranty, and is finished in durable oil‑rubbed bronze.

Input tokens: 446
Output tokens: 114
Cost: 0.007 cents


### 7. Comparing with a Free Local Model (Ollama + Llama 3.2)

Can we do this locally for **$0.00**? Yes! Let's test the same prompt with `ollama/llama3.2` running on localhost:

#### 🔍 Observations & Tradeoffs:
1. **Cost**: Exactly $0.00.
2. **Format Adherence**: Notice minor formatting quirks—Llama 3.2 sometimes uses markdown headers (`#`) instead of the literal `Title:` prefix, and assigns a hyper-specific category (`home security` instead of a broader category).
3. **Token Count Nuance**: LiteLLM reports 48 output tokens for Llama 3.2 vs. 125 tokens for `gpt-oss-20b`. Visually, both summaries are nearly the same length! The difference is that `gpt-oss-20b` is a reasoning model, and LiteLLM includes **internal reasoning tokens** in its token count.
4. **Throughput**: While free, processing 22,000 items locally on a consumer GPU takes significant time. For massive datasets, cloud batch mode is far faster and more reliable.


In [ ]:

messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": items[0].full}]
response = completion(messages=messages, model="ollama/llama3.2", api_base="http://localhost:11434")
print(response.choices[0].message.content)
print()
print(f"Input tokens: {response.usage.prompt_tokens}")
print(f"Output tokens: {response.usage.completion_tokens}")
print(f"Cost: {response._hidden_params['response_cost']*100:.3f} cents")


---
## 8. The Architecture of Batch Mode

Even at 0.011 cents per item, sequentially calling an API 820,000 times in a standard Python loop is impractical:
- Synchronous calls would take days or weeks.
- Network hiccups, rate limits, and dropped sockets would interrupt execution.
- Full price would total ~$90 for the full dataset.

### 💰 The 50% Batch Mode Discount
AI providers (Groq, OpenAI, Anthropic) offer **Batch APIs** that run requests asynchronously within a flexible 24-hour window. Because providers can schedule these jobs during off-peak GPU capacity, **they discount batch inference by 50%**!

### 📄 The JSONL Standard
Batch requests are formatted as a **JSON Lines (`.jsonl`)** file, where each line is an independent, complete JSON object describing one API call:
- `custom_id`: Our item ID string (e.g., `"0"`).
- `method`: `"POST"`.
- `url`: `"/v1/chat/completions"`.
- `body`: Contains the model name, messages list (`system` and `user`), and parameters (`reasoning_effort: "low"`).


In [ ]:
MODEL = "openai/gpt-oss-20b"


Let's define `make_jsonl(item)` to serialize a single `Item` into a valid JSONL request string:


In [ ]:
def make_jsonl(item):
    body = {"model": MODEL, "messages": [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": item.full}], "reasoning_effort": "low"}
    line = {"custom_id": str(item.id), "method": "POST", "url": "/v1/chat/completions", "body": body}
    return json.dumps(line)

Let's inspect the JSONL serialization of `items[0]` to see what a batch line looks like:


In [ ]:
items[0]

In [2]:
make_jsonl(items[0])

NameError: name 'make_jsonl' is not defined

### 9. Generating the Prototype Batch File

We write a helper function `make_file(start, end, filename)` that iterates through a slice of items and writes their JSONL lines to disk.
Let's build a prototype batch file containing the first 1,000 items (`jsonl/0_1000.jsonl`):


In [1]:

def make_file(start, end, filename):
    batch_file = filename
    with open(batch_file, "w", encoding="utf-8") as f:
        for i in range(start, end):
            f.write(make_jsonl(items[i]))
            f.write("\n")

In [ ]:
make_file(0, 1000, "jsonl/0_1000.jsonl")

---
## 10. The 4-Step Batch Workflow with the Groq SDK

Groq's Batch API adopts the exact same standard as OpenAI's Batch API. The lifecycle consists of **4 simple steps**:

```mermaid
flowchart LR
    A["1. files.create<br/>(Upload JSONL)"] --> B["2. batches.create<br/>(Submit 24h Batch)"]
    B --> C["3. batches.retrieve<br/>(Poll Status)"]
    C --> D["4. files.content<br/>(Download Results)"]
```

Let's initialize the Groq client:


In [ ]:
import os
from groq import Groq

groq = Groq(api_key=os.environ.get("GROQ_API_KEY"))

### Step 1: Upload the JSONL File (`files.create`)
We upload `jsonl/0_1000.jsonl` with `purpose="batch"`. Groq stores the file and returns a unique `file_id`:


In [ ]:

with open("jsonl/0_1000.jsonl", "rb", encoding="utf-8") as f:
    response = groq.files.create(file=f, purpose="batch")
response

Let's record the `file_id`:


In [ ]:
file_id = response.id
file_id

### Step 2: Submit the Batch Job (`batches.create`)
We submit the batch job specifying:
- `completion_window="24h"`: Grants the provider up to 24 hours to schedule the job, unlocking the 50% discount.
- `endpoint="/v1/chat/completions"`: Standard chat completion endpoint.
- `input_file_id`: The ID of our uploaded JSONL file.
Groq enqueues the job and returns a `batch_id`:


In [ ]:
response = groq.batches.create(completion_window="24h", endpoint="/v1/chat/completions", input_file_id=file_id)
response

### Step 3: Poll the Batch Status (`batches.retrieve`)
We retrieve the status of our batch. While Groq has 24 hours, small batches of 1,000 items typically complete in **seconds to minutes**!
When finished, `status` switches to `"completed"` and an `output_file_id` is assigned:


In [ ]:
result = groq.batches.retrieve(response.id)
result

### Step 4: Download the Results File (`files.content`)
Once completed, we download the raw output content using `result.output_file_id` and write it to `jsonl/batch_results.jsonl`:


In [ ]:
response = groq.files.content(result.output_file_id)
response.write_to_file("jsonl/batch_results.jsonl")

---
## 11. Parsing Batch Results & Handling Asynchronous Ordering

### ⚠️ The Non-Sequential Order Trap
Because Groq processes batch requests concurrently across a distributed farm of LPUs, **responses in `batch_results.jsonl` do not arrive in the order they were submitted**.

We cannot simply assign line 0 to item 0. Instead, we read `custom_id` from each returned line, convert it to an integer index, and assign the summary directly to `items[id].summary`:


In [ ]:
with open("jsonl/batch_results.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        json_line = json.loads(line)
        id = int(json_line["custom_id"])
        summary = json_line["response"]["body"]["choices"][0]["message"]["content"]
        items[id].summary = summary


Let's compare the original raw text of `items[0]` with its newly populated `.summary`:


In [ ]:
print(items[0].full)

Let's check item index 1000. Notice that it prints `None`! That is because our prototype batch only processed indices 0 through 999:


In [ ]:
print(items[1000].summary)

---
## 12. Scaling Up with the `Batch` Class (`pricer.batch.Batch`)

Now that we have validated the 4-step batch workflow, we want to run this across the entire dataset (all 22,000 items in Lite mode, or all 820,000 items in Full mode).

To make this clean, reproducible, and robust, all of this logic is packaged into [`pricer/batch.py`](file:///C:/Amit/work/code/AI/llm_engineering/myself/Week6/pricer/batch.py):
- **Chunking (`BATCH_SIZE = 1_000`)**: Divides the population into batches of 1,000 items each. Keeping files at 1,000 items ensures fast turnaround, isolates potential errors, and provides real-time progress bars.
- **Folder Management**: Stores generated request files in `lite/batches/` and completed outputs in `lite/output/`.
- **Orchestration**:
  - `Batch.create()`: Generates all JSONL files.
  - `Batch.run()`: Uploads files and submits all batches with a `tqdm` progress bar.
  - `Batch.fetch()`: Polls status and downloads output files as they complete.

### 💰 Cost Summary
- **Lite Dataset (22,000 items)**: Under **$1.00** total on Groq.
- **Full Dataset (820,000 items)**: Under **$30.00** total on Groq.
- **Pre-computed Alternative ($0)**: You can skip running these cells and load Ed Donner's pre-processed datasets directly!

Let's generate the 22 batch files (22 × 1,000 = 22,000 items):


In [ ]:
Batch.create(items, LITE_MODE)

Now we submit all 22 batches to Groq. You will see a `tqdm` progress bar as each batch is uploaded and initiated:


In [ ]:
Batch.run()

We monitor and collect the results using `Batch.fetch()`.
You can call `Batch.fetch()` periodically until all 22 batches report completed and their outputs are downloaded and applied to `items`:


In [ ]:
Batch.fetch()

---
## 13. Quality Assurance & Summary Verification

Let's verify that every single item in our dataset has a non-empty summary.
If no index numbers print out, it means 100% of our items were successfully summarized:


In [ ]:
for index, item in enumerate(items):
    if not item.summary:
        print(index)

Let's inspect item index 10,234 (a red barn photography backdrop) to admire the quality of the LLM-generated summary. Notice how clean, structured, and informative it is compared to raw scraped text:


In [ ]:
print(items[10234].summary)

---
## 14. Data Sanitization & Memory Optimization

Now that we have dense, structured summaries for every item, **we no longer need the raw 4,000-character descriptions (`.full`) or the temporary integer IDs (`.id`)**.

### 📉 Why wipe out these fields?
1. **Dramatic Size Reduction**: Dropping the long `.full` string slashes dataset file size by **over 75%**! For the full 820,000-item dataset, this shrinks the Parquet file down to just **~250MB**, enabling fast network downloads and minimal RAM usage.
2. **Preventing Context Leaks**: Discarding raw boilerplate ensures downstream models only train on clean, standardized summaries.

Let's nullify these fields across all items:


In [ ]:
# Remove the fields that we don't need in the hub

for item in items:
    item.full = None
    item.id = None

---
## 15. Pushing the Final Pre-processed Datasets to Hugging Face Hub

We now partition our pre-processed items into standard splits and push them to Hugging Face Hub:
- **Lite Mode**:
  - `train`: 20,000 items
  - `validation`: 1,000 items
  - `test`: 1,000 items
- **Full Mode**:
  - `train`: 800,000 items
  - `validation`: 10,000 items
  - `test`: 10,000 items
  - (Also pushes the Lite subset so both variants are readily accessible)

The `Item.push_to_hub()` method serializes these partitions into a Hugging Face `DatasetDict` and uploads them to your profile:
*(Leave `username = "ed-donner"` if using the public course datasets!)*


In [ ]:
username = "ed-donner"
full = f"{username}/items_full"
lite = f"{username}/items_lite"

if LITE_MODE:
    train = items[:20_000]
    val = items[20_000:21_000]
    test = items[21_000:]
    Item.push_to_hub(lite, train, val, test)
else:
    train = items[:800_000]
    val = items[800_000:810_000]
    test = items[810_000:]
    Item.push_to_hub(full, train, val, test)

    train_lite = train[:20_000]
    val_lite = val[:1_000]
    test_lite = test[:1_000]
    Item.push_to_hub(lite, train_lite, val_lite, test_lite)

---
## 16. Summary & Key Takeaways from Day 2

### ✅ What We Accomplished:
1. **Semantic Pre-processing**: Used an LLM to rewrite noisy e-commerce product listings into concise, high-signal, colon-delimited summaries.
2. **Token Efficiency**: Cut token counts from ~450 tokens down to ~120 tokens per item while preserving all critical pricing signals.
3. **Batch Mode Mastery**: Implemented the standard 4-step Batch API workflow (`files.create` -> `batches.create` -> `batches.retrieve` -> `files.content`), securing a **50% cost discount**.
4. **Asynchronous Order Resolution**: Handled non-deterministic batch response ordering using integer `custom_id` mapping.
5. **Modular Automation**: Packaged the chunking and execution pipeline into [`pricer.batch.Batch`](file:///C:/Amit/work/code/AI/llm_engineering/myself/Week6/pricer/batch.py) with real-time `tqdm` tracking.
6. **Dataset Footprint Optimization**: Stripped redundant raw text, shrinking the 820,000-item dataset down to ~250MB for efficient storage and fast loading.

---
### 🌐 Public Pre-processed Datasets on Hugging Face Hub
You can inspect the final public datasets directly on Hugging Face:
- 📦 [ed-donner/items_lite](https://huggingface.co/datasets/ed-donner/items_lite): The lightweight 22,000-item dataset.
- 📦 [ed-donner/items_full](https://huggingface.co/datasets/ed-donner/items_full): The full 820,000-item dataset (~250MB).

---
### 🚀 Coming Up in Day 3: Baseline Models & Traditional ML
Tomorrow we take our pre-processed dataset and begin the modeling phase:
1. **The First Rule of Machine Learning**: Why you must *always* build a simple baseline before jumping to complex LLMs or deep learning.
2. **Traditional ML Benchmarks**: Training Dummy Baselines, Linear Regression, Random Forests, and XGBoost on tabular features (Bag of Words, text length, product weight).
3. **Establishing the Gold Standard**: Measuring Mean Absolute Error (MAE) and Median Error so we have an exact target to beat when we train neural networks and fine-tune LLMs in Days 4 and 5!
